# 2. Od relewantnego fragmentu do kodu D

[![Otwórz w Colabie](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caqdastm/ai_qda-workshop-1u/blob/main/04_vibe_coding/02_od_relewantnego_fragmentu_do_kodu_D.ipynb)

Cel: oddzielić decyzję o relewancji od przypisania 0-n procesualnych kodów D.

To jest ćwiczenie z **projektowania pipeline'u kodowania AI_QDA**.
Nie odtwarza autorskiego generatora pełnego wyniku. Kod techniczny jest
zwinięty; widoczne pozostają materiał, karta procedury, odpowiedzi modelu
i decyzja badacza.

W promptach **CZĘŚĆ BADAWCZA** pochodzi z karty uczestnika, a
**DODATEK TECHNICZNY** tylko dopasowuje jedną funkcję do notebooka.


## Rytm pracy

`pytanie i soczewka → procedura relewancji → dwa prompty → powrót do
cytatów → kandydackie D → kontrola struktury → decyzja badacza`


## Dwa porządki pracy — nie mieszamy ich

**1. Tworzenie kodu:** krótką funkcję projektujesz w czacie AI
zintegrowanym z Colabem. Wysyłasz tam instrukcję o procedurze i
kontrakcie funkcji, a otrzymany kod wklejasz do wskazanej komórki.

**2. Analiza materiału:** dopiero działający notebook wysyła prompty i
ograniczony pakiet fragmentów przez API. W formularzu możesz wybrać
`gemini` albo `openai`; dalsze komórki pozostają takie same.

Dla wybranego providera dodaj w Colab Secrets tylko odpowiedni klucz:
`GEMINI_API_KEY` albo `OPENAI_API_KEY`. Przy OpenAI pole
`OPENAI_STORE` pozostaje jawną decyzją uczestnika; adapter zapisze jego
wartość i identyfikator odpowiedzi w lokalnym logu przebiegu.

Tryb `mock` sprawdza przepływ bez wysyłania danych. Czat Colaba służy do
vibe codingu, a `analysis_api` wyłącznie do porównań analitycznych.


In [ ]:
# @title Infrastruktura warsztatu — uruchom bez edycji { display-mode: "form" }
%pip install -q pandas "google-genai>=2.0.0" "openai>=2.0.0"

from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import display

REPOSITORY_SLUG = "caqdastm/ai_qda-workshop-1u" # @param {type:"string"}
repo_folder = REPOSITORY_SLUG.replace("/", "__")
REPO_ROOT = Path("/content") / repo_folder
if not REPO_ROOT.exists():
    subprocess.run(
        ["git", "clone", "-q", f"https://github.com/{REPOSITORY_SLUG}.git", str(REPO_ROOT)],
        check=True,
    )
%cd $REPO_ROOT
support_dir = REPO_ROOT / "04_vibe_coding"
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from workshop_support import (
    AnalysisAPI,
    load_dataframe,
    load_workshop_packet,
    procedure_prompt,
    save_dataframe,
    save_json,
)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    WORKSPACE = Path("/content/drive/MyDrive/AI_QDA_Workshop")
except Exception:
    WORKSPACE = REPO_ROOT / "06_outputs" / "uczestnicy" / "AI_QDA_Workshop"
WORKSPACE.mkdir(parents=True, exist_ok=True)
print("Katalog przekazania między blokami:", WORKSPACE)


In [ ]:
# @title API analityczne — zmiana providera nie zmienia dalszych komórek { display-mode: "form" }
PROVIDER = "mock" # @param ["mock", "gemini", "openai"]
GEMINI_MODEL = "gemini-3.6-flash" # @param {type:"string"}
OPENAI_MODEL = "gpt-5.4-mini" # @param {type:"string"}
OPENAI_STORE = True # @param {type:"boolean"}
AUTHORIZE_API_CALLS = False # @param {type:"boolean"}
MAX_API_CALLS = 2 # @param {type:"integer"}

analysis_api = AnalysisAPI(
    provider=PROVIDER,
    gemini_model=GEMINI_MODEL,
    openai_model=OPENAI_MODEL,
    openai_store=OPENAI_STORE,
    authorize_api_calls=AUTHORIZE_API_CALLS,
    max_api_calls=MAX_API_CALLS,
)
print("Provider:", PROVIDER, "| model:", analysis_api.model, "| limit:", MAX_API_CALLS)


In [ ]:
# @title Wczytaj rejestr z bloku 1 { display-mode: "form" }
packet_path = WORKSPACE / "01_unit_register.csv"
packet = load_dataframe(packet_path) if packet_path.exists() else load_workshop_packet()
display(packet[["text_unit_id", "case_id", "text"]])


In [ ]:
# @title Karta procedury kodowania D { display-mode: "form" }
pytanie_badawcze = "Jak osoby badane opisują doświadczanie niepewności pracy i sposoby reagowania na nią?" # @param {type:"string"}
rama_problemu = "Niepewność pracy, dochodu, czasu, praw i kontroli oraz związane z nimi strategie." # @param {type:"string"}
kryterium_relewantnosci = "Fragment wnosi informację o warunkach niepewności, ich konsekwencjach, ocenach albo działaniach osoby." # @param {type:"string"}
rola_kodu_D = "Krótka procesualna parafraza bliska temu, co dzieje się w dokładnym fragmencie." # @param {type:"string"}
decyzja_badacza = "Czy fragment jest relewantny i czy nazwa D trafnie opisuje jego funkcję w materiale." # @param {type:"string"}
przewidywanie = "Precyzyjne kryterium ograniczy kodowanie ogólnych opisów pracy bez związku z niepewnością." # @param {type:"string"}

PROCEDURE_CARD = {
    "goal": "Wybrać relewantne fragmenty i zapisać 0-n kandydackich D z dokładnym śladem dowodowym.",
    "input": "Rejestr jednostek, pytanie badawcze, rama i kryterium relewantności.",
    "observable_result": "Tabela przypisań z ID fragmentu, kandydackim D, memo i statusem review.",
    "automatic_check": "Istnieją ID, cytat jest dosłowny, status jest dozwolony, a pusta decyzja trafia do review.",
    "researcher_decision": decyzja_badacza,
}


## Vibe coding w czacie Colaba: `check_d_assignments`

1. Uruchom następną komórkę, aby wyświetlić instrukcję.
2. Otwórz panel czatu AI w Colabie i wklej całą instrukcję.
3. Poproś najpierw o krótkie powtórzenie kontraktu zwykłym językiem,
   a następnie o jedną funkcję — bez przebudowy notebooka.
4. Wklej otrzymaną funkcję do komórki **KOMÓRKA UCZESTNIKA**.

Na tym etapie nie korzystasz z klucza API i nie prosisz API
analitycznego o napisanie kodu.


In [ ]:
# @title Wyświetl instrukcję dla czatu Colaba { display-mode: "form" }
appendix = """Napisz funkcję check_d_assignments(assignments, unit_register).
Zwróć tabelę checklisty. Sprawdź ID, dosłowny cytat, status
candidate/needs_review oraz obecność nazwy dla candidate. Nie oceniaj
relewancji ani trafności nazwy D. Nie zmieniaj danych."""
FUNCTION_PROMPT = procedure_prompt(PROCEDURE_CARD, appendix)
print(FUNCTION_PROMPT)


In [ ]:
# KOMÓRKA UCZESTNIKA: wklej pełną funkcję otrzymaną od modelu.
check_d_assignments = None


## Analiza korpusu przez API

Teraz kod pomocniczy jest już w notebooku. Dwa kolejne wywołania API
dostają ten sam materiał, ale inaczej sformułowane zadania analityczne.
Porównujesz wpływ promptu, a nie SDK providera. Zmiana `gemini` na
`openai` odbywa się wyłącznie w formularzu **API analityczne**.


In [ ]:
# @title Dwa wywołania analityczne: porównaj prompt ogólny i kontraktowy { display-mode: "form" }
material = packet[["text_unit_id", "case_id", "text"]].to_csv(index=False)
prompt_a = f"""Zakoduj poniższe fragmenty dotyczące prekaryjności.
Zaproponuj kody i krótko uzasadnij.\n\n{material}"""
prompt_b = f"""Pytanie: {pytanie_badawcze}
Rama: {rama_problemu}
Najpierw dla każdego ID zdecyduj: relevant/not_relevant/needs_review,
używając kryterium: {kryterium_relewantnosci}
Dopiero dla relevant zaproponuj 0-n kodów D. D ma być: {rola_kodu_D}
Zawsze cytuj dokładny fragment i zachowaj text_unit_id. Wyniki są
kandydackie. Nie nadawaj accepted.\n\n{material}"""
response_a = analysis_api.run_analysis(prompt_a, task_label="02_general_prompt")
response_b = analysis_api.run_analysis(prompt_b, task_label="02_contract_prompt")
display(pd.DataFrame([
    {"wariant": "A — ogólny", "odpowiedź": response_a},
    {"wariant": "B — kontrakt", "odpowiedź": response_b},
]))


## Powrót do materiału i decyzja badacza

Odpowiedzi API są kandydackie. Wróć do cytatów i zapisz własną decyzję
w formularzu poniżej. Checklista może wykryć błąd struktury, ale nie
potwierdza trafności kodu, kategorii ani granicy interpretacji.


In [ ]:
# @title Zapisz cztery decyzje po powrocie do fragmentów { display-mode: "form" }
d1_id = "PREWORK_01_U0107" # @param {type:"string"}
d1_name = "" # @param {type:"string"}
d1_memo = "" # @param {type:"string"}
d2_id = "PREWORK_01_U0119" # @param {type:"string"}
d2_name = "" # @param {type:"string"}
d2_memo = "" # @param {type:"string"}
d3_id = "PREWORK_02_U0088" # @param {type:"string"}
d3_name = "" # @param {type:"string"}
d3_memo = "" # @param {type:"string"}
d4_id = "PREWORK_03_U0136" # @param {type:"string"}
d4_name = "" # @param {type:"string"}
d4_memo = "" # @param {type:"string"}

decisions = [(d1_id, d1_name, d1_memo), (d2_id, d2_name, d2_memo),
             (d3_id, d3_name, d3_memo), (d4_id, d4_name, d4_memo)]
rows = []
packet_by_id = packet.set_index("text_unit_id")
for index, (evidence_id, name, memo) in enumerate(decisions, start=1):
    source = packet_by_id.loc[evidence_id]
    rows.append({
        "assignment_id": f"A{index:02d}",
        "code_id": f"D{index:02d}",
        "text_unit_id": evidence_id,
        "code_name": name.strip(),
        "evidence_quote": source["text"],
        "memo": memo.strip(),
        "review_status": "candidate" if name.strip() else "needs_review",
    })
d_assignments = pd.DataFrame(rows)
display(d_assignments)


In [ ]:
# @title Rozwiązanie awaryjne i checklista { display-mode: "form" }
def prepared_check_d_assignments(assignments, unit_register):
    sources = unit_register.set_index("text_unit_id")["text"].to_dict()
    ids_ok = assignments["text_unit_id"].isin(sources).all()
    quotes_ok = all(sources.get(row.text_unit_id) == row.evidence_quote for row in assignments.itertuples())
    statuses_ok = assignments["review_status"].isin({"candidate", "needs_review"}).all()
    names_ok = assignments.loc[assignments["review_status"].eq("candidate"), "code_name"].astype(str).str.strip().ne("").all()
    return pd.DataFrame([
        {"kontrola": "ID istnieją", "zaliczona": ids_ok},
        {"kontrola": "Cytaty są dosłowne", "zaliczona": quotes_ok},
        {"kontrola": "Statusy są przed review", "zaliczona": statuses_ok},
        {"kontrola": "Candidate ma nazwę D", "zaliczona": names_ok},
    ])
if not callable(globals().get("check_d_assignments")):
    check_d_assignments = prepared_check_d_assignments
checklist = check_d_assignments(d_assignments, packet)
display(checklist)
print("Kontrola nie ocenia relewancji ani trafności kodu D.")


In [ ]:
# @title Zapisz artefakty i przekazanie do bloku 3 { display-mode: "form" }
save_dataframe(WORKSPACE / "02_d_assignments.csv", d_assignments)
save_json(WORKSPACE / "02_procedure_card.json", PROCEDURE_CARD)
save_json(WORKSPACE / "02_prompt_prediction.json", {
    "prediction": przewidywanie,
    "research_question": pytanie_badawcze,
    "analytic_frame": rama_problemu,
})
analysis_api.export_runs(WORKSPACE / "02_prompt_runs.jsonl")
print("Zapisano blok 2 w", WORKSPACE)


## Handoff

Blok 3 nie grupuje samych nazw. Ładuje D razem z cytatami i memo,
aby pytać o wspólny mechanizm, granicę oraz przypadek negatywny.
